In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/dhruvildave/en-fr-translation-dataset/en-fr.csv
/kaggle/input/datasets/devicharith/language-translation-englishfrench/eng_-french.csv


In [2]:
import torch
import matplotlib.pyplot as plt
import random
import math
import torch.nn.functional as F
import torch.nn as nn

In [3]:
lines = open('/kaggle/input/datasets/devicharith/language-translation-englishfrench/eng_-french.csv', 'r',  encoding='utf-8').read().splitlines()
len(lines)

175622

In [4]:
lines = [line.replace('\u202f', ' ') for line in lines]

data = [(line.split(',')) for line in lines]
# data[:10]

data = data[1:]

data[:10]



[['Hi.', 'Salut!'],
 ['Run!', 'Cours !'],
 ['Run!', 'Courez !'],
 ['Who?', 'Qui ?'],
 ['Wow!', 'Ça alors !'],
 ['Fire!', 'Au feu !'],
 ['Help!', "À l'aide !"],
 ['Jump.', 'Saute.'],
 ['Stop!', 'Ça suffit !'],
 ['Stop!', 'Stop !']]

In [5]:
X = [d[0] for d in data]
Y = [d[1] for d in data]




In [ ]:
rnn = nn.RNN(10, 20, 2)
input = torch.randn(5, 3, 10)
h0 = torch.randn(2, 3, 20)
output, hn = rnn(input, h0)
output

In [6]:
import re

special_tokens = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]

src_word2idx = {}
src_idx2word = {}

tgt_word2idx = {}
tgt_idx2word = {}


# initialize special tokens
for idx, token in enumerate(special_tokens):

    src_word2idx[token] = idx
    src_idx2word[idx] = token

    tgt_word2idx[token] = idx
    tgt_idx2word[idx] = token


def tokenize(sentence):

    sentence = sentence.lower().strip()

    # split punctuation into separate tokens
    tokens = re.findall(r"\w+(?:'\w+)*|[^\w\s]", sentence)

    return tokens


# build english vocab
current_idx = len(special_tokens)

for sentence in X:

    tokens = tokenize(sentence)

    for token in tokens:

        if token not in src_word2idx:

            src_word2idx[token] = current_idx
            src_idx2word[current_idx] = token

            current_idx += 1


# build french vocab
current_idx = len(special_tokens)

for sentence in Y:

    tokens = tokenize(sentence)

    for token in tokens:

        if token not in tgt_word2idx:

            tgt_word2idx[token] = current_idx
            tgt_idx2word[current_idx] = token

            current_idx += 1

In [7]:
print(len(src_word2idx))
list(tgt_word2idx.keys())[:20]

14221


['<PAD>',
 '<SOS>',
 '<EOS>',
 '<UNK>',
 'salut',
 '!',
 'cours',
 'courez',
 'qui',
 '?',
 'ça',
 'alors',
 'au',
 'feu',
 'à',
 "l'aide",
 'saute',
 '.',
 'suffit',
 'stop']

In [8]:
class RNNCell(nn.Module):

    def __init__(self, input_size, hidden_size):
        super().__init__()

        # bias = false
        self.hidden_size = hidden_size

        # we will initialise the weights of a single RNNCell
        # Wx and Wh - weights of the input and the hidden layer

        # assume, input_dim = 4, hidden = 5
        # input x_t at any time = (4, )
        # output has to be 5, so Wx has to be (5, 4)

        # h = (hidden_size, )
        # Wh must be (hidden_size, hidden_size)
        

        self.Wx = nn.Parameter(torch.randn(hidden_size, input_size))
        self.Wh = nn.Parameter(torch.randn(hidden_size, hidden_size))


    def forward(self, x_t, h_prev):

        if h_prev is None:
            h_prev = torch.zeros(self.Wh.shape[0], )

        h_t = torch.tanh(self.Wx @ x_t + self.Wh @ h_prev)
        return h_t

In [9]:
class RNN(nn.Module):

    def __init__(
        self,
        input_size,
        hidden_size,
        num_layers = 1
    ):

        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.cells = nn.ModuleList(RNNCell(hidden_size, hidden_size) for _ in range(num_layers))
        # first layer alone has input x while all the others have output of prev layers
        self.cells[0] = RNNCell(input_size, hidden_size)


    def forward(
        self,
        x,
        h
    ):
        time_steps = x.shape[0] # x is of shape (time_steps, input_dims)
        
        outputs = []

        if h is None:
            h = torch.zeros(self.num_layers, self.hidden_size) # 3, 5

        
        for t in range(time_steps):

            x_t = x[t]
            h_t_1 = x_t
            
            for layer in range(self.num_layers):

                current_cell = self.cells[layer]

                # h_t_1 is of size (input_size, ) for the first layer
                # h_t_1 is of size (hidden, ) for subsequent layers
                # h_layer is of size (hidden_size, )
                
                h_t = current_cell(h_t_1, h[layer])
                # num_layers, hidden_size
                # so the input must be of size hidden
                h_t_1 = h_t
                h[layer] = h_t
                

            outputs.append(h_t)

        return {"outputs": torch.stack(outputs), "final_hidden_state": h} # array of h across all timestamps               
                

        
        
        

In [10]:
"""
Update gate:
z_t = σ(W_z x_t + U_z h_(t-1))

Reset gate:
r_t = σ(W_r x_t + U_r h_(t-1))

Candidate hidden state:
h~_t = tanh(W_h x_t + U_h (r_t ⊙ h_(t-1)))

Final hidden state:
h_t = (1 - z_t) ⊙ h_(t-1) + z_t ⊙ h~_t
"""

# We are implementing the GRU Cell here

class GRUCell(nn.Module):

    def __init__(self, input_size, hidden_size):
        super().__init__()

        # bias = false
        self.hidden_size = hidden_size
        self.input_size = input_size

        """

            The grucell will take the following as input: x_t and h_t-1 
            we need the following 6 weights W_z, Uz, Wr, Ur, Wh, Uh 
            
            dimensions:
                x_t is going to be (input_dim, 1) 
                h_t-1 is going to be (hidden, 1) 
            
            so Wz should be (hiddenxinput_dim) 
            Uz sohuld be (hiddenxhidden)
            Wr should be same as Wz and 
            Ur same as Uz 
            same for Wh as well
            
        """
        

        self.Wz = nn.Parameter(torch.randn(hidden_size, input_size))
        self.Uz = nn.Parameter(torch.randn(hidden_size, hidden_size))

        self.Wr = nn.Parameter(torch.randn(hidden_size, input_size))
        self.Ur = nn.Parameter(torch.randn(hidden_size, hidden_size))

        self.Wh = nn.Parameter(torch.randn(hidden_size, input_size))
        self.Uh = nn.Parameter(torch.randn(hidden_size, hidden_size))


    def forward(self, x_t, h_prev):

        if h_prev is None:
            h_prev = torch.zeros(self.Wh.shape[0], )

        """
            Update gate:
            z_t = σ(W_z x_t + U_z h_(t-1))
            
            Reset gate:
            r_t = σ(W_r x_t + U_r h_(t-1))
            
            Candidate hidden state:
            h~_t = tanh(W_h x_t + U_h (r_t ⊙ h_(t-1)))
            
            Final hidden state:
            h_t = (1 - z_t) ⊙ h_(t-1) + z_t ⊙ h~_t
        """

        z_t = torch.sigmoid(self.Wz @ x_t + self.Uz @ h_prev)
        r_t = torch.sigmoid(self.Wr @ x_t + self.Ur @ h_prev)
        h_candidate_t = torch.tanh(self.Wh @ x_t + self.Uh @ (r_t * h_prev))
        h_t = (torch.ones(z_t.shape) - z_t) * h_prev + z_t * h_candidate_t

        return h_t


In [11]:
input = torch.randn(2)
torch.sigmoid(input)


tensor([0.8979, 0.6972])

In [25]:
class GRU(nn.Module):

    def __init__(
        self,
        input_size,
        hidden_size,
        num_layers = 1
    ):

        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.cells = nn.ModuleList(GRUCell(hidden_size, hidden_size) for _ in range(num_layers))
        # first layer alone has input x while all the others have output of prev layers
        self.cells[0] = GRUCell(input_size, hidden_size)


    def forward(
        self,
        x,
        h = None
    ):
        time_steps = x.shape[0] # x is of shape (time_steps, input_dims)
        
        outputs = []

        if h is None:
            h = torch.zeros(self.num_layers, self.hidden_size) # 3, 5

        
        for t in range(time_steps):

            x_t = x[t]
            h_t_1 = x_t
            new_hidden_states = []
            
            for layer in range(self.num_layers):

                current_cell = self.cells[layer]

                # h_t_1 is of size (input_size, ) for the first layer
                # h_t_1 is of size (hidden, ) for subsequent layers
                # h_layer is of size (hidden_size, )
                
                h_t = current_cell(h_t_1, h[layer])
                # num_layers, hidden_size
                # so the input must be of size hidden
                h_t_1 = h_t
                new_hidden_states.append(h_t)

            h = torch.stack(new_hidden_states)
            outputs.append(h_t)

        return {"outputs": torch.stack(outputs), "final_hidden_state": h} # array of h across all timestamps               
                

        
        
        

In [26]:
class Encoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_size,
        num_layers = 1
            ):

        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.gru = GRU(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers
        )

    

    def forward(self, x):
        embedded = self.embedding(x) # (seq_length, num_dims)

        outputs = self.gru(embedded)
        return outputs

In [27]:
X[111450], Y[111450]
print(len(src_word2idx))
print(len(tgt_idx2word))

print(X[100].split(" "))

14221
28671
['Come', 'in.']


In [28]:
x = X[1000]
tokens = tokenize(x)
print(tokens)
token_ids = torch.tensor([src_word2idx[token] for token in tokens]) # (seq_length, )

encoder = Encoder(14221, 4, 50, 3)
op = encoder(token_ids)

print(op["outputs"].shape)
op["final_hidden_state"].shape


["we're", 'shy', '.']
torch.Size([3, 50])


torch.Size([3, 50])

In [29]:
class BahdanauAttention(nn.Module):
    
    def __init__(self, hidden_size):
        super().__init__()
        self.Wh = nn.Parameter(torch.randn(hidden_size, hidden_size))
        self.Ws = nn.Parameter(torch.randn(hidden_size, hidden_size))
        self.v = nn.Parameter(torch.randn(hidden_size,))

    def forward(self, decoder_hidden, encoder_outputs):

        alignments = []

        for encoder_op in encoder_outputs:
            tanh = torch.tanh(self.Wh @ encoder_op + self.Ws @ decoder_hidden)
            e_t_i = self.v.T @ tanh
            alignments.append(e_t_i)


        alignments = torch.stack(alignments) # shape (-1, )
        attention_weights = alignments.softmax(dim=0)
        attention_weights = attention_weights.view(-1, 1)

        context_vector = torch.sum(attention_weights * encoder_outputs, dim=0)

        return {"context": context_vector, "attention_weights": attention_weights}

        
        
        
            

In [30]:
class Decoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_size,
        num_layers=1
    ):

        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.attention = BahdanauAttention(hidden_size)

        self.gru = GRU(
            input_size=embedding_dim + hidden_size,
            hidden_size = hidden_size,
            num_layers = num_layers
        )

        self.output_layer = nn.Linear(hidden_size, vocab_size)
        

    def forward(self, encoded_outputs, decoder_prev_hidden, prev_output):
        s_t_1 = decoder_prev_hidden
        h = encoded_outputs
        y_t_1 = prev_output

        # we need the previous output as part of the current input
        x_t = self.embedding(y_t_1) # -> shape (embedded_dim, )

        # bahdanau attention takes encoded_outputs, decoder_prev_hidden
        # this returns context vector and attention vector

        attention_output = self.attention(decoder_prev_hidden[-1], encoded_outputs)

        attention_wts = attention_output.get("attention_weights")
        context_vector = attention_output.get("context")

        combined_input = torch.cat((x_t, context_vector), dim=0)

        combined_input = combined_input.view(1, -1)

        decoder_gru_op = self.gru(combined_input, decoder_prev_hidden)
        
        op = decoder_gru_op.get("outputs")
        decoder_final_hidden = decoder_gru_op.get("final_hidden_state")

        logits = self.output_layer(op)

        return (logits, decoder_final_hidden, attention_wts)

        
        
        

        
            


In [ ]:
attention_wts_collected = []

In [32]:
class Seq2Seq(nn.Module):

    def __init__(
        self,
        encoder,
        decoder,
        tgt_word2idx
    ):

        super().__init__()

        self.encoder = encoder
        self.decoder = decoder

        self.sos_token = tgt_word2idx["<SOS>"]
        self.eos_token = tgt_word2idx["<EOS>"]


    def forward(
        self,
        src,
        tgt
    ):

        encoder_op = self.encoder(src)

        encoder_outputs = encoder_op.get("outputs")

        encoder_hidden = encoder_op.get(
            "final_hidden_state"
        )


        # initialize decoder hidden state
        decoder_hidden = encoder_hidden


        # first decoder input = <SOS>
        prev_output = torch.tensor(self.sos_token)


        outputs = []


        # teacher forcing loop
        for t in range(len(tgt)):

            decoder_op = self.decoder(
                encoder_outputs,
                decoder_hidden,
                prev_output
            )

            logits, decoder_hidden, attention_wts = decoder_op
            attention_wts_collected.append(attention_wts.view(-1, ))

            outputs.append(logits)


            # teacher forcing:
            # use actual target token
            prev_output = tgt[t]


        outputs = torch.cat(outputs, dim=0)

        return outputs

In [33]:
encoder = Encoder(
    vocab_size=len(src_word2idx),
    embedding_dim=64,
    hidden_size=128,
    num_layers=2
)

decoder = Decoder(
    vocab_size=len(tgt_word2idx),
    embedding_dim=64,
    hidden_size=128,
    num_layers=2
)

model = Seq2Seq(
    encoder,
    decoder,
    tgt_word2idx
)

In [35]:
epoch_losses = []

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

loss_function = nn.CrossEntropyLoss()



num_epochs = 50


for epoch in range(num_epochs):

    total_loss = 0

    for src_sentence, tgt_sentence in zip(X[:100], Y[:100]):

        src_tokens = tokenize(src_sentence)
        tgt_tokens = tokenize(tgt_sentence)


        # add special tokens
        tgt_tokens = ["<SOS>"] + tgt_tokens + ["<EOS>"]


        # get the token IDs
        src_ids = torch.tensor([
            src_word2idx[token]
            for token in src_tokens
        ])


        tgt_ids = torch.tensor([
            tgt_word2idx[token]
            for token in tgt_tokens
        ])


        # forward pass
        
        optimizer.zero_grad()

        logits = model(
            src_ids,
            tgt_ids[:-1]
        )


        # logits:
        # (target_seq_len - 1, vocab_size)

        targets = tgt_ids[1:]


        loss = loss_function(
            logits,
            targets
        )


        #backward pass
        loss.backward()
        optimizer.step()

        total_loss += loss.item()


    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")
    epoch_losses.append(total_loss)

Epoch 1, Loss: 313.8659
Epoch 2, Loss: 319.2800
Epoch 3, Loss: 309.5030
Epoch 4, Loss: 282.3504
Epoch 5, Loss: 264.1948
Epoch 6, Loss: 261.6377
Epoch 7, Loss: 249.7288
Epoch 8, Loss: 240.2441
Epoch 9, Loss: 219.3656
Epoch 10, Loss: 219.1783
Epoch 11, Loss: 209.4969
Epoch 12, Loss: 208.5425
Epoch 13, Loss: 244.4627
Epoch 14, Loss: 217.0054
Epoch 15, Loss: 220.1956
Epoch 16, Loss: 225.7786
Epoch 17, Loss: 204.8684
Epoch 18, Loss: 198.4819
Epoch 19, Loss: 184.6496
Epoch 20, Loss: 198.5048
Epoch 21, Loss: 187.8600
Epoch 22, Loss: 205.7654
Epoch 23, Loss: 204.4819
Epoch 24, Loss: 206.2914
Epoch 25, Loss: 190.6243
Epoch 26, Loss: 188.5451
Epoch 27, Loss: 178.9348
Epoch 28, Loss: 181.8432
Epoch 29, Loss: 170.7746
Epoch 30, Loss: 178.0306
Epoch 31, Loss: 179.2081
Epoch 32, Loss: 164.6981
Epoch 33, Loss: 169.3815
Epoch 34, Loss: 159.2117
Epoch 35, Loss: 158.1194
Epoch 36, Loss: 161.6472
Epoch 37, Loss: 159.2444
Epoch 38, Loss: 175.1053
Epoch 39, Loss: 152.5008
Epoch 40, Loss: 161.5184
Epoch 41,

In [37]:
def translate(
    model,
    sentence,
    max_len=20
):

    model.eval()

    src_tokens = tokenize(sentence)

    src_ids = torch.tensor([ src_word2idx.get(token, src_word2idx["<UNK>"])
        for token in src_tokens
    ])


    # encoder phase
    encoder_op = model.encoder(src_ids)

    encoder_outputs = encoder_op["outputs"]

    decoder_hidden = encoder_op["final_hidden_state"]


    # decoder phase
    current_token = torch.tensor(
        tgt_word2idx["<SOS>"]
    )

    generated_tokens = []


    for _ in range(max_len):

        logits, decoder_hidden, attention_wts = (
            model.decoder(encoder_outputs, decoder_hidden,current_token)
        )


        predicted_token_id = torch.argmax(logits.squeeze(0)).item()
        predicted_word = tgt_idx2word[predicted_token_id]

        if predicted_word == "<EOS>":
            break


        generated_tokens.append(predicted_word)

        current_token = torch.tensor(predicted_token_id)

    return " ".join(generated_tokens)

In [42]:
translate(model, "Run!")

'cours'